# Biorthogonal Basis Decomposition

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import scipy.sparse as sparse

from tomotok.geometry import RegularGrid, generate_sightlines, sparse_line
from tomotok.inversions import Bob, CholeskySolver, SparseInvSolver
from tomotok.tools.phantoms import regular_elliptical_flux, gaussian_on_flux

### Phantom and Grid

In [ ]:
nr, nz = 50, 60
grid = RegularGrid(nr, nz, rlims=(0.2, 0.7), zlims=(-0.3, 0.3))
flux = regular_elliptical_flux(grid)
phantom = gaussian_on_flux(flux, amplitude=2000, width=0.05, center=0.3)
phantom = np.roll(phantom, (3, -5), (0, 1))  # move the phantom from the center of the grid

In [ ]:
plt.figure()
plt.axes().set_aspect('equal')
plt.pcolormesh(*grid.border_mesh, phantom)
plt.colorbar(label='Emissivity [-]')
plt.xlabel('r [m]')
plt.ylabel('Z [m]')
plt.show()

### Artificial Camera Setup

In [ ]:
pinhole = (0.8, -0.1, 0.2)
resolution = (60, 60)
# resolution = (800, 800)  # higher resolutions requires significant amount of memory
fov = (40, 40)
axis = (-1, -0.3, -0.3)

### Geometry Matrix

In [ ]:
start, end = generate_sightlines(pinhole=pinhole, num=resolution, fov=fov, axis=axis, length=3)
gmat = sparse_line(start, end, grid, step=1e-4)

### Synthetic Image

In [ ]:
image = gmat @ phantom.flatten()  # noiseless flattened image obtained from phantom
# adding gaussian noise
# reconstructions using simple basis can deteriorate when noise is included
# escale = 10
# image = image + np.random.normal(0, escale, image.size).reshape(image.shape)

plt.figure()
plt.axis('off')
plt.imshow(image.reshape(resolution))
plt.colorbar(label='Signal [-]')
plt.show()

## Decomposition
Has to be done only once for a given basis and geometry matrix

Basis matrix should hold base vectors (simple, wavelets...) in columns

the choice of basis influences method's de-noising capability

In [ ]:
basis = sparse.eye_array(gmat.shape[1], format='csc')  # simplest choice, no de-noising capability

The decomposition requires the inverse of a matrix computed by a solver of linear systems, which is selected using the `engine` parameter of `Bob`:

 - `CholeskySolver` (default) - dense Cholesky decomposition, robust and fast for smaller and/or dense matrices
 - `SparseInvSolver` - sparse inverse from SciPy, faster for large and highly sparse matrices
 - `SksparseCholesky` from `tomotok.inversions.solvers.sksparse` - sparse Cholesky decomposition useful for large matrices, requires the optional scikit-sparse package

The `reg_factor` parameter adds a small multiple of the identity matrix, which ensures the positive definiteness required by the Cholesky decomposition. This regularisation can also suppress oscillations caused by noise in insufficiently viewed areas of the reconstruction plane.

In [ ]:
engine_ch = CholeskySolver()  # robust, faster for smaller and/or dense matrices
engine_sp = SparseInvSolver()  # faster for large and highly sparse matrices
bob = Bob(engine=engine_ch)

bob.decompose(gmat, basis, reg_factor=1e-10)

## Inversion

In [ ]:
res = bob(image)

In [ ]:
plt.figure()
plt.imshow(res.reshape(grid.shape), origin='lower', extent=grid.extent, vmin=-phantom.max(), vmax=phantom.max())
plt.colorbar(label='Emissivity [-]')
plt.xlabel('R [-]')
plt.ylabel('z [-]')
# plt.savefig('/figure/location/name.suffix')
plt.show()

In [ ]:
# retrofit
rf = gmat @ res.T
plt.figure()
plt.imshow((rf-image).reshape(resolution))
plt.colorbar(label='Retrofit - Signal [-]')
plt.axis('off')
plt.show()

## Thresholded

In [ ]:
rest = bob.thresholding(image, c=3.1)

In [ ]:
plt.figure()
plt.imshow(rest.reshape(grid.shape), origin='lower', extent=grid.extent)
plt.xlabel('R [-]')
plt.ylabel('z [-]')
plt.colorbar(label='Emissivity [-]')
plt.show()

In [ ]:
plt.figure()
# plt.imshow(phantom - rest.reshape(grid.shape), origin='lower', extent=grid.extent)
plt.imshow((res - rest).reshape(grid.shape), origin='lower', extent=grid.extent)
plt.xlabel('R [-]')
plt.ylabel('z [-]')
plt.colorbar(label='Result - Thresholded [-]')
plt.show()

In [ ]:
rft = gmat @ rest  # retrofit, forward model

In [ ]:
plt.figure()
plt.axis('off')
plt.imshow((rft-image).reshape(resolution))
plt.colorbar(label='Retrofit - Signal [-]')
plt.show()

### Total Recorded Intensity Check

In [ ]:
print(f'Signal sum: {image.sum():.1f} for original image')
print(f'Signal sum: {rf.sum():.1f} for full inversion')
print(f'Signal sum: {rft.sum():.1f} for thresholded inversion')

## Saving decomposed matrices

In [ ]:
# bob.save_decomposition('path/to/decomposed_mat')
# bob2 = Bob()
# bob2.load_decomposition('path/to/decomposed_mat')